# Requirement 2 · Datasets in Google Colab

This notebook downloads the Kaggle classroom dataset, maps its labels to *attentive / distracted / inactive*,
makes a **leakage-safe** train / validation / test split, and shows the label statistics.

**No GPU is needed for this step** (Runtime → Change runtime type → CPU is fine and saves your GPU quota).

## 0 · Put the project on Google Drive (once)

1. Zip the `smart Attendance` folder on your laptop **without** `data/`, `models/` and `reports/`
   (they are rebuilt here). Rename the zip to `smart_attendance.zip`.
2. Upload it to `MyDrive/` in Google Drive.

Keeping data and results on Drive means they survive when Colab disconnects (Colab's own disk is wiped).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
PROJECT = '/content/drive/MyDrive/smart_attendance'
if not os.path.exists(PROJECT):
    zipfile.ZipFile('/content/drive/MyDrive/smart_attendance.zip').extractall(PROJECT)
# if the zip contained a top-level folder, step into it
inner = [d for d in os.listdir(PROJECT) if os.path.isfile(os.path.join(PROJECT, d, 'pyproject.toml'))]
if inner:
    PROJECT = os.path.join(PROJECT, inner[0])
%cd "$PROJECT"
!ls

In [ ]:
# Colab already has torch, torchvision and OpenCV; this adds the rest and installs our package
!pip install -q -r requirements.txt ultralytics kagglehub
!pip install -q -e .

## 1 · Download from Kaggle

`kagglehub` is Kaggle's official download library. This dataset is public, so **no Kaggle account or API key
is needed**. The download is 129 MB.

Dataset page: https://www.kaggle.com/datasets/mustafaxgm/classroom-student-engagement-dataset
(licence CC BY-NC 4.0: free for non-commercial academic work, with attribution).

In [ ]:
import kagglehub
kaggle_path = kagglehub.dataset_download('mustafaxgm/classroom-student-engagement-dataset')
print(kaggle_path)
!ls "$kaggle_path/dataset"

## 2 · Labels, de-duplication and the leakage-safe split

`prepare()` does everything in one call:

* reads the YOLO labels (one line per student: class, centre x, centre y, width, height, all 0-1)
* maps the 8 actions to 3 states (`look_forward/read/write/handrise/stand` → attentive,
  `turn_head/using_device` → distracted, `sleep` → inactive)
* finds near-duplicate frames with a perceptual hash and removes any that would cross splits
* splits by **recording group** (same students, same room) so no student appears in two splits
* writes three versions: people-only YOLO labels (counting), 3-state YOLO labels (detector training),
  and per-student crops (ResNet / GoogLeNet classification)

In [ ]:
from smartattend.data.kaggle_classroom import prepare, make_figures
report = prepare('data/raw', 'data/processed/kaggle_classroom', source=kaggle_path)
print('duplicates:', report['duplicates'])
for split, s in report['splits'].items():
    print(f"{split:5s} {s['images']:4d} images {s['people']:5d} people  groups={s['groups']}  states={s['states']}")

**Expected output**

```
duplicates: {'near_duplicate_pairs_within_group': 50, 'removed_across_groups': 0}
train  217 images  2277 people  groups=['misc', 'owais', 'seminar']  states={'attentive': 1759, 'distracted': 448, 'inactive': 70}
val    179 images  1167 people  groups=['taimoor-class']  states={'attentive': 954, 'distracted': 143, 'inactive': 70}
test    85 images  1159 people  groups=['taimoor-zaeem']  states={'distracted': 646, 'attentive': 421, 'inactive': 92}
```

The 50 near-duplicates are consecutive video frames *inside* one group, so they stay in one split and cannot leak.

In [ ]:
from IPython.display import Image, display
for p in make_figures('data/processed/kaggle_classroom', 'reports/figures/data'):
    display(Image(str(p)))

## 3 · The other datasets (optional here)

| purpose | dataset | how |
|---|---|---|
| face recognition, fairness | LFW: https://www.kaggle.com/datasets/jessicali9530/lfw-dataset | `kagglehub.dataset_download('jessicali9530/lfw-dataset')` (118 MB), or `python -m smartattend stage data` (Hugging Face mirror with identity labels) |
| person detection on classroom-like photos | COCO 2017 val (official site, not Kaggle: the Kaggle copy is 27 GB) | `python -m smartattend stage data` downloads only the 250 MB of annotations and about 600 selected images |
| alertness frames with timestamps | drowsiness faces (Hugging Face, MIT) | `python -m smartattend stage data` |

Storage: everything together is about 1 GB, well within Colab's limits.

In [ ]:
!python -m smartattend stage data      # COCO-classroom + LFW + drowsiness (about 10 minutes)
!du -sh data/raw/* data/processed/*